In [9]:
import os
import sys
os.environ["KERAS_BACKEND"] = "torch"
sys.path.append("./src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
import keras
from keras.metrics import Recall, Precision
from sklearn.metrics import confusion_matrix

from models import *
from utils import *


# -------------------
# CONFIG
# -------------------
DATA_ROOT = "processed_data"
Multi_scale_patch = False
IMAGE_FOLDER = os.path.join(DATA_ROOT, "images")
LABELS_FILE = os.path.join(DATA_ROOT, "labels.csv")



CONFIG = {
    "img_size": (64, 64, 1),
    "batch_size": 32,
    "epochs": 20,
    "learning_rate": 1e-4,
    "model_name": "mobilenetv2_transfer",   # 👈 change this to switch model
}


In [10]:
# -------------------
# HELPERS
# -------------------
def compile_model(model, config):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss='binary_crossentropy',
        metrics=['accuracy', Recall(name="recall"), Precision(name="precision")]
    )
    return model

In [11]:
# -------------------
# LOAD LABELS
# -------------------
df = pd.read_csv(LABELS_FILE)

print("Total samples:", len(df))
print(df['label'].value_counts())

# -------------------
# SPLIT
# -------------------
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

# -------------------
# LOAD INTO MEMORY
# -------------------
X_train, Y_train = load_data(train_df, DATA_ROOT, Multi_scale_patch)
X_val, Y_val = load_data(val_df, DATA_ROOT, Multi_scale_patch)
X_test, Y_test = load_data(test_df, DATA_ROOT, Multi_scale_patch)


if Multi_scale_patch:
    print("Train balance:", np.mean(Y_train))
    print("Train:", X_train[0].shape)
    print("Train:", X_train[1].shape)
    print("Val:", X_val[0].shape)
    print("Val:", X_val[1].shape)
    print("Test:", X_test[0].shape)
    print("Test:", X_test[1].shape)
else:
    print("Train:", X_train.shape)
    print("Val:", X_val.shape)
    print("Test:", X_test.shape)


Total samples: 48460
label
1    24230
0    24230
Name: count, dtype: int64
Train: (33922, 64, 64, 1)
Val: (7269, 64, 64, 1)
Test: (7269, 64, 64, 1)


In [12]:
# -------------------
# Model training
# -------------------
# build model
model = get_model(CONFIG["model_name"],CONFIG["img_size"]) 

# compile
model = compile_model(model, CONFIG)

model.summary()

# train
if Multi_scale_patch:
    history = model.fit(
        [X_train[0],X_train[1]], Y_train,
        validation_data=([X_val[0],X_val[1]], Y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
    )
    y_pred_probs = model.predict([X_test[0],X_test[1]])
else:
    history = model.fit(
    X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
    )
    y_pred_probs = model.predict(X_test)
# get predictions (probabilities)


# convert to binary (threshold = 0.5)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = Y_test.astype(int)

# -------------------
# PER-LABEL ACCURACY
# -------------------
for label in [0, 1]:
    idx = (y_true == label)
    
    if np.sum(idx) == 0:
        print(f"Label {label}: no samples")
        continue

    acc = np.mean(y_pred[idx] == y_true[idx])
    print(f"Accuracy for label {label}: {acc:.4f} (n={np.sum(idx)})")

cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

# pretty print
tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")
# test
model.evaluate(X_test, Y_test)

c:\STABLE DIFFUESION MY 2025\DAT255 Git colab\DAT255-V26-\src\models.py:387: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 64, 64, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 64, 64, 3) │          0 │ input_layer_4[0]… │
│ (Concatenate)       │                   │            │ input_layer_4[0]… │
│                     │                   │            │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 64, 64, 3) │          0 │ concatenate_2[0]… │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 64, 64, 3) │          0 │ rescaling_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 2, 2,      │  2,257,984 │ lambda_2[0][0]    │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │    163,968 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │        129 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/20
1061/1061 ━━━━━━━━━━━━━━━━━━━━ 66s 63ms/step - accuracy: 0.6355 - loss: 0.6450 - precision: 0.6320 - recall: 0.6490 - val_accuracy: 0.6844 - val_loss: 0.5904 - val_precision: 0.7076 - val_recall: 0.6286
Epoch 2/20
1061/1061 ━━━━━━━━━━━━━━━━━━━━ 73s 69ms/step - accuracy: 0.6786 - loss: 0.5904 - precision: 0.6750 - recall: 0.6890 - val_accuracy: 0.6949 - val_loss: 0.5753 - val_precision: 0.6995 - val_recall: 0.6834
Epoch 3/20
1061/1061 ━━━━━━━━━━━━━━━━━━━━ 72s 68ms/step - accuracy: 0.6950 - loss: 0.5718 - precision: 0.6906 - recall: 0.7063 - val_accuracy: 0.6945 - val_loss: 0.5668 - val_precision: 0.7052 - val_recall: 0.6685
Epoch 4/20
1061/1061 ━━━━━━━━━━━━━━━━━━━━ 68s 64ms/step - accuracy: 0.7023 - loss: 0.5614 - precision: 0.6990 - recall: 0.7104 - val_accuracy: 0.7020 - val_loss: 0.5624 - val_precision: 0.7150 - val_recall: 0.6721
Epoch 5/20
1061/1061 ━━━━━━━━━━━━━━━━━━━━ 73s 69ms/step - accuracy: 0.7113 - loss: 0.5513 - precision: 0.7068 - recall: 0.7221 - val_accuracy: 0

[0.5693945288658142,
 0.7094510793685913,
 0.7228949069976807,
 0.7039121389389038]

In [ ]:
# -------------------
# CONFIG (adjust these)
# -------------------
Load_from_ZIP = True
zip_path = "data/vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0.zip"
image_folder = "data/train_images/"
patch_size = (128, 128)
stride = 16

# -------------------
# LOAD CSV (must exist already)
# -------------------
df = pd.read_csv("data/finding_annotations.csv")
df = df[['image_id','finding_categories','xmin','ymin','xmax','ymax']]

# -------------------
# BUILD IMAGE INDEX
# -------------------
image_index, z = build_image_index(
    image_folder=image_folder,
    zip_path=zip_path,
    load_from_zip=Load_from_ZIP
)

# -------------------
# HEATMAP GENERATION
# -------------------
def generate_heatmap(model, img):
    h, w = img.shape
    heatmap = np.zeros((h, w))
    counts = np.zeros((h, w))

    for y in range(0, h - patch_size[0] + 1, stride):
        for x in range(0, w - patch_size[1] + 1, stride):

            if Multi_scale_patch:
                p_small, p_large = extract_multiscale_patch(img, x, y, patch_size[0])

                p_small = np.expand_dims(p_small, axis=(0, -1))
                p_large = np.expand_dims(p_large, axis=(0, -1))

                pred = model.predict([p_small, p_large], verbose=0)[0][0]

            else:
                patch = img[y:y+patch_size[0], x:x+patch_size[1]]
                patch = np.expand_dims(patch, axis=(0, -1))

                pred = model.predict(patch, verbose=0)[0][0]

            # optional threshold stabilization
            pred = 0 if pred < 0.3 else pred

            heatmap[y:y+patch_size[0], x:x+patch_size[1]] += pred
            counts[y:y+patch_size[0], x:x+patch_size[1]] += 1

    heatmap = heatmap / (counts + 1e-6)

    return heatmap

# -------------------
# VISUALIZATION
# -------------------
def show_heatmap(model, image_id):
    img = load_image(
    image_id=image_id,
    image_index=image_index,
    load_from_zip=Load_from_ZIP,
    zip_file=z
)

    if img is None:
        print("Image not found")
        return

    heatmap = generate_heatmap(model, img)

    plt.figure(figsize=(10,10))
    plt.imshow(img, cmap='gray')

    # overlay heatmap
    plt.imshow(heatmap, cmap='jet', alpha=0.4)

    # draw GT boxes
    rows = df[df['image_id'] == image_id]

    for _, row in rows.iterrows():
        if row['finding_categories'] == "['Mass']" and not np.isnan(row['xmin']):
            x = row['xmin']
            y = row['ymin']
            w = row['xmax'] - row['xmin']
            h = row['ymax'] - row['ymin']

            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor='lime',
                facecolor='none'
            )
            plt.gca().add_patch(rect)

    plt.title(f"Heatmap for {image_id}")
    plt.axis('off')
    plt.show()

def get_random_mass_image_id(df):
    # keep only rows with Mass and valid boxes
    mass_df = df[
        (df['finding_categories'] == "['Mass']") &
        (~df['xmin'].isna())
    ]

    if len(mass_df) == 0:
        print("No Mass images found in dataframe")
        return None

    # get unique image_ids
    image_ids = mass_df['image_id'].unique()

    # pick one randomly
    return np.random.choice(image_ids)
# -------------------
# RUN (random image)
# -------------------
image_id = get_random_mass_image_id(df)



if image_id is not None:
    print("Selected image:", image_id)
    show_heatmap(model, image_id)